In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")[:10]
OPENAI_API_KEY

'sk-proj-2t'

In [3]:
# 필수 import v1.0
from langchain_openai.chat_models.base import ChatOpenAI
from langchain_openai.llms.base import OpenAI
from langchain_core.output_parsers.base import BaseOutputParser
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.prompts.chat import ChatMessagePromptTemplate, ChatPromptTemplate

In [4]:
chat = ChatOpenAI()

In [ ]:
chat.invoke("메시가 최고의 선수인 이유는?")

# temperature (창의력)
- 0.0 ~ 0.3 : 결정론 - 요약, 추출, 일관성, 정확성
- 0.5 ~ 0.7 : 균형적 = 적당히 자연스러움, 유연성
- 0.8 ~ 1.0+ : 무작위 - 창의적이고 예측 불가능

In [6]:
chat = ChatOpenAI(temperature = 0)
result = chat.invoke("하늘은 무슨색이니?")
result.content

'하늘은 파란색이다.'

In [7]:
chat = ChatOpenAI(temperature = 0.5)
result = chat.invoke("하늘은 무슨색이니?")
result.content

'하늘은 주로 푸른색이다. 하지만 날씨나 시간에 따라 하늘의 색깔은 변할 수 있다. 일출이나 일몰 시간에는 주로 붉은색이나 주황색으로 변할 수 있고, 흐린 날씨에는 회색 또는 흰색으로 보일 수도 있다.'

In [10]:
chat = ChatOpenAI(temperature = 1.0)
result = chat.invoke("하늘은 무슨색이니?")
result.content

'하늘은 대개 파란색을 가지고 있지만, 일출 또는 일몰 시간대에는 주로 붉은빛이 반사되어 핑크 또는 오렌지빛으로 변할 수도 있습니다. 때로는 흰색 구름으로 가령에 가려져서 회색빛을 띠기도 합니다. 하늘의 색은 시간과 조명 등 여러 가지 요소에 따라 계속 변화하므로, 항상 동일한 색상을 가지고 있지는 않습니다.'

In [12]:
chat = ChatOpenAI(temperature = 1.0)
result = chat.invoke("메시와 호날두중 누가 최고의 선수야? , 둘중 한명을 너가 무조건 골라")
result.content

'아 정말 선택하기 어렵군요. 그러나 제 생각에는 메시가 최고의 선수라고 생각합니다.'

In [14]:
chat = ChatOpenAI(temperature = 1.0)
result = chat.invoke("25-26시즌 EPL 우승팀은 어디야?")
result.content

'25-26시즌 EPL 우승팀은 리버풀입니다.'

In [22]:
class NewLineOutputParser(BaseOutputParser):
    # 반드시 parse
    def parse(self, text):
        lines = text.split("\n")
        return [line.lstrip("-123456789. ").strip() for line in lines]

In [23]:
newline_parser = NewLineOutputParser()

In [24]:
newline_parser.parse("""- 1. 햄버거\n- 2. 떡볶이\n- 3. 치킨""")

['햄버거', '떡볶이', '치킨']

In [26]:
template = ChatPromptTemplate.from_messages([
    ("system", """
        리스트를 생성하는 기계입니다.
        요청한 모든 리스트에 개수는 최대 {max_length}개 까지만 목록으로 표시하세요.
        그 이상 초과되는 리스트는 답변하지 마세요.
    """),
    ("human", "{question}")
])

prompt = template.format_messages(
    max_length = 5,
    question = "AI를 잘하려면 어떤것부터 공부해야해?"
)

# 체인(Chain 생성)
- "|": 파이프 연산자로 체인을 만든다

In [27]:
# list

first_chain = template | chat | NewLineOutputParser()

In [28]:
chain_result = first_chain.invoke({
    "max_length" : 5,
    "question": "AI를 잘하려면 어떤것부터 공부해야해?"
})

In [29]:
chain_result

['수학: 선형대수, 미적분, 확률 및 통계 등',
 '프로그래밍 언어: Python, R, Java, C++ 등',
 '머신러닝 알고리즘: 회귀분석, 의사결정나무, 나이브 베이즈, 신경망 등',
 '데이터 전처리: 데이터 수집, 정제, 변환, 특성공학 등',
 '모델평가 및 성능향상: 과적합 방지, 교차검증, 하이퍼파라미터 튜닝 등']

In [30]:
# RunnableSequence
# 이전 단계의 출력이 다음 단계의 입력으로 자동 전달되는 파이프라인 객체
print(type(first_chain))

<class 'langchain_core.runnables.base.RunnableSequence'>


# 1. 템플릿 생성

In [36]:
template = ChatPromptTemplate.from_messages([
    ("system", """
        당신은 세계적인 수준의 여행 가이드입니다.
        사람들이 좋아하는 여행 장소를 많이 알고 있습니다.
        설명없이 지역 명소 이름만 목록으로 {max_length}개 까지 답변하세요.
        목록의 개수가 초과하는 것은 답변하지 마세요.
    """),
    ("human", """
        {place} 여행 장소 추천해줘!
    """)
])

# 2.

In [37]:
second_chain = template | chat

In [38]:
trip_result = second_chain.invoke({
    "max_length":3,
    "place":"부산"
})

In [40]:
trip_result

AIMessage(content='1. 해운대해수욕장\n2. 부산 국립과학관\n3. 벡스코 (벡스코 국제컨벤션센터)', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 132, 'total_tokens': 185, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DecPn6R38iA9W6QSK6NnVCf4Ob6k3', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1b38-a75d-77a1-928b-fee71b74b2bf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 132, 'output_tokens': 53, 'total_tokens': 185, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# 실습

In [ ]:
# 저녁 메뉴 추천(양식, 중식, 한식 선택할 수 있도록)
# 체인을 생성 후 결과를 출력

In [41]:
template = ChatPromptTemplate.from_messages([
    ("system", """
        당신은 한국에 유명한 미식가입니다,
        설명없이 음식 메뉴 이름만 목록으로 {max_length}개 까지 답변하세요.
        목록의 개수가 초과하는 것은 답변하지 마세요.
    """),
    ("human", """
        {menu} 중 음식을 추천해줘!
    """)
])

In [46]:
menu_chain = template | chat

## .stream() 
- 결과를 실시간 chunk 단위로 쪼개서 응답한다

In [47]:
menu_result = menu_chain.stream({
    "max_length": 5,
    "menu" : "양식"
})
    

In [49]:
for chunk in menu_result:
    print(chunk.content, end="", flush = True)

1. 스테이크
2. 파스타
3. 그라탕
4. 라자냐
5. 샐러드

In [45]:
menu_result.content

'1. 스테이크\n2. 파스타\n3. 그라탕\n4. 레몬 버터 연어\n5. 비프 스튜'

In [ ]:
# 형태로 딕셔너리
# 최근 급등한 주식 리스트 5개

In [54]:
template = ChatPromptTemplate.from_messages([
    ("system", """
        넌 주식의 마스터야 
        사용자가 주식과 관련된 질문을 하면 {max_length}개의 주식을 추천해줘

        반드시 아래의 형식으로 답변해줘
        출력 형식:
        {{
            "one" : "tesla",
            "two" : "samsung",
            ...
            "five" : "apple"
        }}
    """),
    ("human","{question}")
])
        

In [55]:
stock_chain = template | chat

In [56]:
stock_result = stock_chain.invoke({
    "max_length" : 5,
    "question" : "최근 급등한 미국 주식 추천해줘!"
})

In [57]:
stock_result.content

'{\n    "one": "Tesla",\n    "two": "Amazon",\n    "three": "Zoom Video Communications",\n    "four": "Sea Limited",\n    "five": "Nvidia"\n}'

In [ ]:
import json

json.loads(stock_result.

In [ ]:
class JsonOutputParser(BaseOutputParser):
    def parser(self,text):
        return json.loads(text)

In [ ]:
chat = ChatOpenAI(temperature=0)

stock_chain = template | chat | JsonOutputParser()

In [ ]:
stock_result = stock_chain.invoke({
    "max_length" : 5,
    "question" : "최근 급등한 미국 주식 추천해줘!"
})

In [ ]:
stock_result